# Optuna: Hyperparameter Optimization Framework

## What Is Optuna?

Imagine tuning a race car engine. You could try every combination of settings (GridSearch — exhaustive but slow),  
or randomly try combinations (RandomSearch — faster but dumb).  
**Optuna** is an expert mechanic: it tries a setting, sees how the car performs, and uses that knowledge to make  
smarter guesses for the next setting — converging on the best configuration much faster.

**Optuna** is a hyperparameter optimization (HPO) framework that uses **Bayesian optimization** to find the best  
hyperparameters for any ML model or function.

Key features:
- **Define-by-run API**: define search space inside the objective function — very flexible
- **TPE sampler**: Tree-structured Parzen Estimator — samples smarter than random
- **Pruning**: stop bad trials early (like early stopping for HPO)
- **Visualization**: rich plots for trial analysis
- **Distributed**: run trials in parallel across machines
- **Framework agnostic**: works with sklearn, PyTorch, TensorFlow, LightGBM, XGBoost

## Resources

- **Docs**: [https://optuna.readthedocs.io/](https://optuna.readthedocs.io/)
- **GitHub**: [https://github.com/optuna/optuna](https://github.com/optuna/optuna)
- **YouTube**: [https://www.youtube.com/watch?v=P6NwZVl8ttc](https://www.youtube.com/watch?v=P6NwZVl8ttc)
- **Paper**: [Optuna: A Next-generation Hyperparameter Optimization Framework](https://arxiv.org/abs/1907.10902)

## Installation

```bash
pip install optuna
# Optional: visualization
pip install optuna[visualization]  # plotly
# Optional: dashboard
pip install optuna-dashboard
```

In [ ]:
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress trial logs
    OPTUNA_AVAILABLE = True
    print(f"Optuna version: {optuna.__version__}")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("Optuna not installed — simulated output shown.")
    print("Install: pip install optuna")

from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Synthetic dataset
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nDataset: {X_train.shape[0]} train, {X_test.shape[0]} test, {X.shape[1]} features")

## Core Concept 1: The Objective Function

Optuna's key idea: wrap your model training in an **objective function** that:
1. Samples hyperparameters using `trial.suggest_*()`
2. Trains the model with those hyperparameters
3. Returns a score (Optuna minimizes — return negative AUC to maximize AUC)

In [ ]:
if OPTUNA_AVAILABLE:
    def objective(trial):
        """Objective function: define search space, train model, return score."""

        # Define hyperparameter search space
        n_estimators   = trial.suggest_int('n_estimators', 50, 500)
        max_depth      = trial.suggest_int('max_depth', 2, 15)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        max_features   = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

        # Train model with suggested hyperparameters
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            max_features=max_features,
            random_state=42, n_jobs=-1,
        )

        # 3-fold CV for robust evaluation
        scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)
        return scores.mean()  # Optuna maximizes (we set direction='maximize')

    # Create study (a collection of trials)
    study = optuna.create_study(
        direction='maximize',    # maximize AUC
        study_name='rf_tuning',
        sampler=optuna.samplers.TPESampler(seed=42),  # Bayesian (TPE)
    )

    print("Running 30 trials with TPE sampler...")
    t0 = time.time()
    study.optimize(objective, n_trials=30, show_progress_bar=False)
    elapsed = time.time() - t0

    print(f"Optimization complete in {elapsed:.1f}s")
    print(f"\nBest trial:")
    print(f"  Value (AUC): {study.best_value:.4f}")
    print(f"  Params:      {study.best_params}")

    # Train final model with best params
    best_rf = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=-1)
    best_rf.fit(X_train, y_train)
    test_auc = roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1])
    print(f"  Test AUC:    {test_auc:.4f}")

    # Compare to default RandomForest
    default_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
    default_rf.fit(X_train, y_train)
    default_auc = roc_auc_score(y_test, default_rf.predict_proba(X_test)[:, 1])
    print(f"\nDefault RF AUC: {default_auc:.4f}")
    print(f"Optuna RF AUC:  {test_auc:.4f}  (+{(test_auc-default_auc)*100:.2f}% improvement)")

else:
    print("Optuna objective function (simulated):")
    print()
    print("  def objective(trial):")
    print("      n_estimators = trial.suggest_int('n_estimators', 50, 500)")
    print("      max_depth    = trial.suggest_int('max_depth', 2, 15)")
    print("      max_features = trial.suggest_categorical('max_features', ['sqrt','log2'])")
    print()
    print("      model = RandomForestClassifier(n_estimators=n_estimators, ...)")
    print("      scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc')")
    print("      return scores.mean()")
    print()
    print("  study = optuna.create_study(direction='maximize')")
    print("  study.optimize(objective, n_trials=30)")
    print()
    print("  Best value: 0.9187")
    print("  Best params: {'n_estimators': 312, 'max_depth': 8, 'max_features': 'sqrt'}")
    print()
    print("  Default RF AUC: 0.9102")
    print("  Optuna RF AUC:  0.9187  (+0.93% improvement)")

## Core Concept 2: Suggest Methods — Defining the Search Space

Optuna's `trial.suggest_*()` methods define what hyperparameters to search and their ranges.

In [ ]:
print("Optuna suggest methods — defining the search space:")
print()

suggest_examples = [
    ('suggest_int',         'trial.suggest_int("n_estimators", 50, 500)',
     'Integer in [50, 500] — uniform'),
    ('suggest_int(step)',   'trial.suggest_int("max_depth", 2, 20, step=2)',
     'Even integers only: 2, 4, 6, ..., 20'),
    ('suggest_float',       'trial.suggest_float("lr", 0.01, 0.3)',
     'Float in [0.01, 0.3] — uniform'),
    ('suggest_float(log)',  'trial.suggest_float("lr", 1e-5, 1e-1, log=True)',
     'Log-scale: explores 1e-5 to 1e-1 evenly in log space'),
    ('suggest_categorical', 'trial.suggest_categorical("optimizer", ["adam", "sgd"])',
     'One of the given choices'),
    ('conditional params',  'if model == "rf": trial.suggest_int("n_trees", 10, 1000)',
     'Dynamic search space based on other choices'),
]

for name, code, desc in suggest_examples:
    print(f"  {name}:")
    print(f"    Code: {code}")
    print(f"    Desc: {desc}")
    print()

print("When to use log=True:")
print("  For learning rates, regularization strengths, etc.")
print("  Without log: uniform(1e-5, 1e-1) → 99% of samples are > 0.001")
print("  With log:    loguniform(1e-5, 1e-1) → equal attention to each order of magnitude")

## Core Concept 3: Multi-Model Search (Conditional Hyperparameters)

Optuna's define-by-run API allows **conditional hyperparameters** — the search space itself can depend on choices.

In [ ]:
if OPTUNA_AVAILABLE:
    def multi_model_objective(trial):
        """Search over multiple model types simultaneously."""

        # Choose which model to try
        model_type = trial.suggest_categorical('model_type', ['rf', 'gbm'])

        if model_type == 'rf':
            # RF-specific hyperparameters
            model = RandomForestClassifier(
                n_estimators=trial.suggest_int('rf_n_estimators', 50, 300),
                max_depth=trial.suggest_int('rf_max_depth', 3, 12),
                random_state=42, n_jobs=-1,
            )
        else:
            # GBM-specific hyperparameters
            model = GradientBoostingClassifier(
                n_estimators=trial.suggest_int('gbm_n_estimators', 50, 300),
                max_depth=trial.suggest_int('gbm_max_depth', 2, 8),
                learning_rate=trial.suggest_float('gbm_lr', 0.01, 0.3, log=True),
                subsample=trial.suggest_float('gbm_subsample', 0.6, 1.0),
                random_state=42,
            )

        scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)
        return scores.mean()

    study2 = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
    print("Searching over RF and GBM simultaneously...")
    study2.optimize(multi_model_objective, n_trials=20)

    print(f"Best model type: {study2.best_params['model_type']}")
    print(f"Best AUC:        {study2.best_value:.4f}")
    print(f"Best params:     {study2.best_params}")

    # Count trials per model type
    import pandas as pd
    trials_df = study2.trials_dataframe()
    if 'params_model_type' in trials_df.columns:
        counts = trials_df['params_model_type'].value_counts()
        print(f"\nTrials per model type:\n{counts}")

else:
    print("Multi-model search (simulated):")
    print()
    print("  def objective(trial):")
    print("      model_type = trial.suggest_categorical('model_type', ['rf', 'gbm'])")
    print()
    print("      if model_type == 'rf':")
    print("          model = RandomForestClassifier(")
    print("              n_estimators=trial.suggest_int('rf_n_estimators', 50, 300),")
    print("          )")
    print("      else:  # gbm")
    print("          model = GradientBoostingClassifier(")
    print("              learning_rate=trial.suggest_float('gbm_lr', 0.01, 0.3, log=True),")
    print("          )")
    print()
    print("  Best model type: gbm")
    print("  Best AUC:        0.9221")
    print("  Best params: {model_type: gbm, gbm_lr: 0.047, gbm_n_estimators: 215, ...}")

## Core Concept 4: Pruning — Kill Bad Trials Early

**Pruning** stops trials that are clearly going to be bad — like early stopping for HPO.  
After each epoch/fold, the pruner checks if this trial is worse than past trials and stops it.

In [ ]:
if OPTUNA_AVAILABLE:
    try:
        import torch
        import torch.nn as nn
        TORCH_AVAILABLE = True
    except ImportError:
        TORCH_AVAILABLE = False

    # Show pruning concept with sklearn (simulate epochs)
    from sklearn.ensemble import GradientBoostingClassifier

    def objective_with_pruning(trial):
        """Prune trial early if intermediate scores are bad."""
        lr    = trial.suggest_float('lr', 0.01, 0.3, log=True)
        depth = trial.suggest_int('depth', 2, 8)

        # Simulate 'epochs' by fitting with different n_estimators
        for n_trees in [20, 50, 100, 200]:
            model = GradientBoostingClassifier(
                n_estimators=n_trees, learning_rate=lr,
                max_depth=depth, random_state=42
            )
            model.fit(X_train, y_train)
            val_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

            # Report intermediate value to the pruner
            trial.report(val_auc, step=n_trees)

            # Check if Optuna wants to stop this trial early
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()  # stop this trial

        return val_auc  # final score if not pruned

    # MedianPruner: prune if trial is below median of past trials at same step
    study_pruned = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    )

    study_pruned.optimize(objective_with_pruning, n_trials=20)

    # Stats
    completed = [t for t in study_pruned.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned    = [t for t in study_pruned.trials if t.state == optuna.trial.TrialState.PRUNED]
    print(f"Pruning results:")
    print(f"  Total trials:    {len(study_pruned.trials)}")
    print(f"  Completed:       {len(completed)}")
    print(f"  Pruned (early):  {len(pruned)}")
    print(f"  Best AUC:        {study_pruned.best_value:.4f}")
    print(f"  Best params:     {study_pruned.best_params}")

else:
    print("Pruning (simulated):")
    print()
    print("  def objective(trial):")
    print("      for n_trees in [20, 50, 100, 200]:  # simulated epochs")
    print("          model.fit(X_train, y_train)")
    print("          val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])")
    print("          trial.report(val_auc, step=n_trees)  # report intermediate")
    print("          if trial.should_prune():")
    print("              raise optuna.exceptions.TrialPruned()  # stop this trial")
    print("      return val_auc")
    print()
    print("  study = optuna.create_study(")
    print("      pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)")
    print("  )")
    print()
    print("  Pruning results:")
    print("    Total trials:   20")
    print("    Completed:       9  (ran all 200 trees)")
    print("    Pruned (early): 11  (stopped at 20 or 50 trees — saved time!)")
    print("    Best AUC:    0.9198")

## Core Concept 5: Study Analysis and Visualization

In [ ]:
if OPTUNA_AVAILABLE:
    import pandas as pd

    # Convert all trials to DataFrame
    trials_df = study.trials_dataframe()
    print("All trials summary:")
    print(trials_df[['number', 'value', 'duration']].describe())
    print()

    # Top 5 trials
    print("Top 5 trials:")
    top5 = trials_df.sort_values('value', ascending=False).head(5)
    print(top5[['number', 'value', 'params_n_estimators', 'params_max_depth']].to_string())
    print()

    # Best params
    print(f"Best trial number: {study.best_trial.number}")
    print(f"Best AUC:          {study.best_value:.4f}")

    # Visualization (requires plotly)
    print()
    print("Optuna visualization functions (require plotly):")
    viz_funcs = [
        ('optuna.visualization.plot_optimization_history(study)',
         'AUC vs trial number — shows convergence'),
        ('optuna.visualization.plot_param_importances(study)',
         'Which hyperparameters matter most'),
        ('optuna.visualization.plot_contour(study, ["max_depth", "n_estimators"])',
         'AUC landscape over two hyperparameters'),
        ('optuna.visualization.plot_slice(study)',
         'AUC vs each hyperparameter (1D slice)'),
    ]
    for fn, desc in viz_funcs:
        print(f"  {fn}")
        print(f"  → {desc}")
        print()

else:
    print("Study analysis (simulated):")
    print()
    print("  trials_df = study.trials_dataframe()")
    print("  # → DataFrame with columns: number, value, duration, params_*")
    print()
    print("  Top trials:")
    print("   trial  value  n_estimators  max_depth")
    print("   17     0.9187  312          8")
    print("   23     0.9181  287          9")
    print("   11     0.9174  334          7")
    print()
    print("  Visualization (requires plotly):")
    print("  optuna.visualization.plot_optimization_history(study)")
    print("    → Shows AUC improving over trials (Bayesian converges faster)")
    print("  optuna.visualization.plot_param_importances(study)")
    print("    → max_depth most important, n_estimators less so")

## Core Concept 6: Samplers — The Brain of Optuna

The **sampler** decides which hyperparameters to try next based on past trial results.

In [ ]:
print("Optuna samplers comparison:")
print()

samplers = [
    {
        'name': 'TPESampler (default)',
        'code': "optuna.samplers.TPESampler(seed=42)",
        'desc': "Tree-structured Parzen Estimator — Bayesian optimization.",
        'how':  "Fits two probability models: good and bad trials. Samples where good/bad ratio is high.",
        'when': "Default choice. Works well for 10-1000 trials.",
    },
    {
        'name': 'RandomSampler',
        'code': "optuna.samplers.RandomSampler(seed=42)",
        'desc': "Pure random search.",
        'how':  "Samples uniformly at random. No learning from past trials.",
        'when': "Good baseline. Surprisingly effective for < 10 parameters.",
    },
    {
        'name': 'CmaEsSampler',
        'code': "optuna.samplers.CmaEsSampler(seed=42)",
        'desc': "Covariance Matrix Adaptation Evolution Strategy.",
        'how':  "Evolutionary: maintains a Gaussian distribution over params, adapts covariance.",
        'when': "Best for continuous parameters with complex correlations. > 20 continuous params.",
    },
    {
        'name': 'GridSampler',
        'code': "optuna.samplers.GridSampler({'lr': [0.01, 0.1], 'depth': [3, 5]})",
        'desc': "Exhaustive grid search.",
        'how':  "Tries all combinations. Same as sklearn GridSearchCV.",
        'when': "Only when you have few hyperparameters with known good ranges.",
    },
]

for s in samplers:
    print(f"  {s['name']}:")
    print(f"    Code: {s['code']}")
    print(f"    How:  {s['how']}")
    print(f"    When: {s['when']}")
    print()

print("Rule of thumb:")
print("  < 5 hyperparameters, small dataset → RandomSampler (fast, good enough)")
print("  5-15 hyperparameters              → TPESampler (default, best overall)")
print("  > 15 continuous hyperparameters   → CmaEsSampler")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Too few trials | Sub-optimal hyperparameters | Use at least 50-100 trials for Bayesian (TPE needs warm-up) |
| No cross-validation in objective | Overfitting to validation set | Always use CV inside objective, not a single train/val split |
| Search space too wide | TPE wanders, doesn't converge | Narrow the range based on domain knowledge |
| Forgetting `n_jobs=-1` inside model | Slow trials | Use parallel model training inside each trial |
| Using pruning without `trial.report()` | Pruner never prunes | Must call `trial.report(val, step=n)` for pruner to act |
| Maximizing when you should minimize | Wrong direction | Check: RMSE → `minimize`, AUC → `maximize` |

## Mini Project: LightGBM HPO Pipeline

In [ ]:
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False

if OPTUNA_AVAILABLE and LGB_AVAILABLE:
    print("=" * 60)
    print("LIGHTGBM HPO WITH OPTUNA")
    print("=" * 60)
    print()

    def lgb_objective(trial):
        params = {
            'objective':        'binary',
            'metric':           'auc',
            'verbosity':        -1,
            'boosting_type':    trial.suggest_categorical('boosting', ['gbdt', 'dart']),
            'num_leaves':       trial.suggest_int('num_leaves', 20, 200),
            'max_depth':        trial.suggest_int('max_depth', 3, 12),
            'learning_rate':    trial.suggest_float('lr', 1e-3, 0.3, log=True),
            'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample', 0.5, 1.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42,
        }
        model = lgb.LGBMClassifier(**params)
        scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)
        return scores.mean()

    lgb_study = optuna.create_study(direction='maximize',
                                     sampler=optuna.samplers.TPESampler(seed=42))
    lgb_study.optimize(lgb_objective, n_trials=50)

    print(f"Best AUC:    {lgb_study.best_value:.4f}")
    print(f"Best params: {lgb_study.best_params}")

    # Final model
    best_lgb = lgb.LGBMClassifier(**lgb_study.best_params, random_state=42, verbosity=-1)
    best_lgb.fit(X_train, y_train)
    final_auc = roc_auc_score(y_test, best_lgb.predict_proba(X_test)[:, 1])
    print(f"Test AUC:    {final_auc:.4f}")

else:
    print("LightGBM HPO with Optuna (simulated):")
    print()
    print("  50 trials with TPE sampler on 9 hyperparameters")
    print()
    print("  Best AUC:    0.9312")
    print("  Best params:")
    print("    boosting:  gbdt")
    print("    num_leaves: 87")
    print("    max_depth: 7")
    print("    lr: 0.0234")
    print("    n_estimators: 312")
    print("    subsample: 0.812")
    print("    colsample: 0.734")
    print("    reg_alpha: 0.0043")
    print("    reg_lambda: 0.187")
    print()
    print("  Test AUC: 0.9298")
    print("  Default LGB AUC: 0.9145  (+1.6% improvement from Optuna!")

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "What is TPE and how does it differ from random search?",
     "a": """TPE = Tree-structured Parzen Estimator. It's a Bayesian optimization algorithm.

Random Search:
  - Samples hyperparameters uniformly at random
  - No learning from past trials
  - 20th trial is just as 'dumb' as the 1st
  - Efficient for: < 5 hyperparameters, parallel budget

TPE (Bayesian):
  1. Split past trials into 'good' (top 25%) and 'bad' (bottom 75%)
  2. Fit a probability density l(x) on good trials
  3. Fit a probability density g(x) on bad trials
  4. Sample where l(x)/g(x) is highest (likely to be good, unlikely to be bad)
  5. Each trial makes the next proposal smarter

TPE advantages:
  - Works well with 5-20 hyperparameters
  - Finds good solutions in fewer trials (10-5× fewer than grid search)
  - Handles categorical + continuous + conditional parameters

When random beats TPE:
  - Very few trials (< 10): not enough data for good density estimation
  - Highly parallel: TPE is sequential; random runs perfectly in parallel"""},

    {"q": "Optuna vs sklearn GridSearchCV vs RandomizedSearchCV — when to use each?",
     "a": """GridSearchCV:
  When: small, discrete search space; you know the good range
  Pro:  exhaustive, guaranteed to find the best in the grid
  Con:  exponential in hyperparameters (3 params × 5 values = 125 trials)
  Use:  2-3 hyperparameters with small known ranges

RandomizedSearchCV:
  When: larger search space; fixed trial budget
  Pro:  simple, parallel, surprisingly effective
  Con:  no learning from past trials; can't handle conditional params
  Use:  quick baseline HPO, 5-10 hyperparameters

Optuna:
  When: many hyperparameters; want best accuracy; need conditional params
  Pro:  Bayesian (smarter), pruning, visualization, distributed
  Con:  slightly more code than sklearn; sequential by default
  Use:  serious HPO, LightGBM/XGBoost/NN tuning, > 5 hyperparameters

Rule:
  Prototype quickly → RandomizedSearchCV
  Best possible model → Optuna with TPE + pruning
  Known small space → GridSearchCV"""},

    {"q": "How does Optuna pruning work and when should you use it?",
     "a": """Pruning stops unpromising trials early — like early stopping for HPO.

How it works:
  1. Trial runs through 'steps' (epochs, CV folds, or tree depths)
  2. At each step, call: trial.report(intermediate_value, step=n)
  3. Pruner checks: is this trial worse than median of past trials at step n?
  4. If yes: trial.should_prune() returns True
  5. Raise optuna.exceptions.TrialPruned() to stop

Pruners:
  MedianPruner: prune if below median at same step (simple, robust)
  HyperbandPruner: based on Hyperband algorithm (good for NN epochs)
  SuccessiveHalvingPruner: run many trials briefly, promote top-K
  NopPruner: no pruning (default if not specified)

When to use:
  - NN training (report val_loss every epoch, prune if diverging)
  - Multi-stage training (report after each CV fold)
  - Expensive trials (GPU hours)

When NOT to use:
  - Training time is trivial (< 1s per trial)
  - No meaningful 'intermediate' metric exists (single train/eval)"""},

    {"q": "How would you use Optuna to tune a PyTorch neural network?",
     "a": """Optuna integrates naturally with PyTorch training loops.

def objective(trial):
    # Search space
    n_layers    = trial.suggest_int('n_layers', 1, 4)
    hidden_size = trial.suggest_int('hidden_size', 32, 512, log=True)
    lr          = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    dropout     = trial.suggest_float('dropout', 0.1, 0.5)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD'])

    # Build model dynamically
    layers = []
    in_size = input_dim
    for i in range(n_layers):
        layers += [nn.Linear(in_size, hidden_size), nn.ReLU(), nn.Dropout(dropout)]
        in_size = hidden_size
    layers += [nn.Linear(hidden_size, 1), nn.Sigmoid()]
    model = nn.Sequential(*layers)

    optimizer = getattr(torch.optim, optimizer_name)(model.parameters(), lr=lr)

    for epoch in range(100):
        train_epoch(model, optimizer, train_loader)
        val_acc = evaluate(model, val_loader)

        trial.report(val_acc, epoch)  # report intermediate
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()  # stop bad trial early

    return val_acc

study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.HyperbandPruner()
)
study.optimize(objective, n_trials=100)"""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Concept | Optuna API |
|---------|------------|
| Create study | `optuna.create_study(direction='maximize')` |
| Run optimization | `study.optimize(objective, n_trials=100)` |
| Integer param | `trial.suggest_int('name', low, high)` |
| Float param | `trial.suggest_float('name', low, high)` |
| Log-scale float | `trial.suggest_float('lr', 1e-5, 1e-1, log=True)` |
| Categorical | `trial.suggest_categorical('opt', ['adam', 'sgd'])` |
| Best value | `study.best_value` |
| Best params | `study.best_params` |
| All trials | `study.trials_dataframe()` |
| Bayesian sampler | `TPESampler(seed=42)` (default) |
| Pruning | `trial.report(val, step=n)` + `trial.should_prune()` |
| Median pruner | `optuna.pruners.MedianPruner()` |
| Visualization | `optuna.visualization.plot_optimization_history(study)` |

### Next Steps
1. **Optuna tutorial**: [https://optuna.readthedocs.io/en/stable/tutorial/](https://optuna.readthedocs.io/en/stable/tutorial/)
2. **Optuna + LightGBM**: [https://github.com/optuna/optuna-examples](https://github.com/optuna/optuna-examples)
3. **Next**: Ray Tune — distributed HPO across a cluster